#Encoder-Decoder Model

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------
# 1. Dataset
# -------------------------
data = [
    ("one", "un"),
    ("two", "deux"),
    ("three", "trois"),
    ("four", "quatre"),
    ("five", "cinq")
]

# -------------------------
# 2. Build Vocabulary
# -------------------------
input_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
target_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}

for src, tgt in data:
    if src not in input_vocab:
        input_vocab[src] = len(input_vocab)
    if tgt not in target_vocab:
        target_vocab[tgt] = len(target_vocab)

input_size = len(input_vocab)
output_size = len(target_vocab)

# Reverse mapping for decoding
target_idx_to_word = {v:k for k,v in target_vocab.items()}

# -------------------------
# 3. Encode Function
# -------------------------
def encode_sentence(sentence, vocab):
    return torch.tensor([
        vocab["<sos>"],
        vocab[sentence],
        vocab["<eos>"]
    ], dtype=torch.long)

# -------------------------
# 4. Encoder
# -------------------------
class Encoder(nn.Module):

    def __init__(self, input_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(input_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size)

    def forward(self, x):

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell

# -------------------------
# 5. Decoder
# -------------------------
class Decoder(nn.Module):

    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(output_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell):

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        predictions = self.fc(outputs.squeeze(0))

        return predictions, hidden, cell

# -------------------------
# 6. Seq2Seq Model
# -------------------------
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target):

        hidden, cell = self.encoder(source)

        outputs = []

        x = torch.tensor([target_vocab["<sos>"]])

        for t in range(1, len(target)):

            output, hidden, cell = self.decoder(x, hidden, cell)

            outputs.append(output)

            x = target[t].unsqueeze(0)  # teacher forcing: Fixed to ensure x is 1D

        outputs = torch.cat(outputs, dim=0)

        return outputs

# -------------------------
# 7. Initialize Model
# -------------------------
embed_size = 16
hidden_size = 32

encoder = Encoder(input_size, embed_size, hidden_size)
decoder = Decoder(output_size, embed_size, hidden_size)

model = Seq2Seq(encoder, decoder)

# -------------------------
# 8. Loss & Optimizer
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------
# 9. Training Loop
# -------------------------
epochs = 300

for epoch in range(epochs):

    total_loss = 0

    for src, tgt in data:

        src_tensor = encode_sentence(src, input_vocab)
        tgt_tensor = encode_sentence(tgt, target_vocab)

        optimizer.zero_grad()

        output = model(src_tensor, tgt_tensor)

        loss = criterion(output, tgt_tensor[1:])

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print("Epoch", epoch, "Loss:", total_loss)

# -------------------------
# 10. Translation Function
# -------------------------
def translate(word):

    model.eval()

    src_tensor = encode_sentence(word, input_vocab)

    hidden, cell = encoder(src_tensor)

    x = torch.tensor([target_vocab["<sos>"]])

    result = []

    for _ in range(5):

        output, hidden, cell = decoder(x, hidden, cell)

        pred = output.argmax(1).item()

        if pred == target_vocab["<eos>"]:
            break

        result.append(target_idx_to_word[pred])

        x = torch.tensor([pred])

    return " ".join(result)

# -------------------------
# 11. Test
# -------------------------
print("one ->", translate("one"))
print("two ->", translate("two"))
print("three ->", translate("three"))

Epoch 0 Loss: 10.3086439371109
Epoch 50 Loss: 1.628286212682724
Epoch 100 Loss: 0.29243309423327446
Epoch 150 Loss: 0.12054833024740219
Epoch 200 Loss: 0.06783730536699295
Epoch 250 Loss: 0.04380459664389491
one -> un
two -> deux
three -> trois


#Transformer Model

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# -----------------------------
# Dataset
# -----------------------------
data = [
    ("one", "un"),
    ("two", "deux"),
    ("three", "trois"),
    ("four", "quatre"),
    ("five", "cinq")
]

# -----------------------------
# Build Vocabulary
# -----------------------------
input_vocab = {"<pad>":0,"<sos>":1,"<eos>":2}
target_vocab = {"<pad>":0,"<sos>":1,"<eos>":2}

for src, tgt in data:
    if src not in input_vocab:
        input_vocab[src] = len(input_vocab)
    if tgt not in target_vocab:
        target_vocab[tgt] = len(target_vocab)

input_size = len(input_vocab)
output_size = len(target_vocab)

target_idx_to_word = {v:k for k,v in target_vocab.items()}

# -----------------------------
# Encode Function
# -----------------------------
def encode(sentence, vocab):
    return torch.tensor([
        vocab["<sos>"],
        vocab[sentence],
        vocab["<eos>"]
    ], dtype=torch.long)

# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(1)

        self.register_buffer("pe", pe)

    def forward(self, x):

        x = x + self.pe[:x.size(0)]

        return x


# -----------------------------
# Transformer Model
# -----------------------------
class TransformerModel(nn.Module):

    def __init__(self, input_size, output_size, d_model=32, nhead=4, num_layers=2):

        super().__init__()

        self.embedding = nn.Embedding(input_size, d_model)

        self.pos_encoder = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers
        )

        self.fc_out = nn.Linear(d_model, output_size)

    def forward(self, src, tgt):

        src = src.unsqueeze(1)
        tgt = tgt.unsqueeze(1)

        src = self.embedding(src)
        tgt = self.embedding(tgt)

        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)

        output = self.transformer(src, tgt)

        output = self.fc_out(output)

        return output.squeeze(1)


# -----------------------------
# Initialize Model
# -----------------------------
model = TransformerModel(input_size, output_size)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# Training
# -----------------------------
epochs = 300

for epoch in range(epochs):

    total_loss = 0

    for src, tgt in data:

        src_tensor = encode(src, input_vocab)

        tgt_tensor = encode(tgt, target_vocab)

        optimizer.zero_grad()

        output = model(src_tensor, tgt_tensor[:-1])

        loss = criterion(output, tgt_tensor[1:])

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print("Epoch", epoch, "Loss:", total_loss)


# -----------------------------
# Translation Function
# -----------------------------
def translate(word):

    model.eval()

    src = encode(word, input_vocab)

    tgt = torch.tensor([target_vocab["<sos>"]])

    for _ in range(5):

        output = model(src, tgt)

        pred = output[-1].argmax().item()

        if pred == target_vocab["<eos>"]:
            break

        tgt = torch.cat((tgt, torch.tensor([pred])))

    result = [target_idx_to_word[i.item()] for i in tgt[1:]]

    return " ".join(result)


# -----------------------------
# Test
# -----------------------------
print("one ->", translate("one"))
print("two ->", translate("two"))
print("three ->", translate("three"))

Epoch 0 Loss: 10.631844878196716
Epoch 50 Loss: 0.19917405024170876
Epoch 100 Loss: 0.08030947111546993
Epoch 150 Loss: 0.030968446750193834
Epoch 200 Loss: 0.017784653697162867
Epoch 250 Loss: 0.013404713477939367
one -> un
two -> deux
three -> trois
